# Análisis de IRR_Results

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_DIR = Path("../data/raw").resolve()
INTERIM_DIR = Path("../data/interim").resolve()

REPO01 = (
    RAW_DIR
    / "01"
    / "28489601"
)

INTERACTIONS_PATH = (
    REPO01
    / "interactions.csv"
)

AGGREGATES_PATH = (
    REPO01
    / "aggregates.csv"
)

print(INTERACTIONS_PATH.exists())
print(AGGREGATES_PATH.exists())

True
True


In [3]:
columnas_interaccion = [
    "id",
    "user_id",
    "timestamp",
    "url",

    "mouse.position.x",
    "mouse.position.y",

    "mouse.clicks",
    "mouse.clicks.left",

    "scroll.absolute.x",
    "scroll.absolute.y",
    "scroll.relative.x",
    "scroll.relative.y",

    "keyboard",
    "keyboard.alpha",
    "keyboard.numeric",
    "keyboard.function",
    "keyboard.symbol"
]

In [4]:
interactions = pd.read_csv(
    INTERACTIONS_PATH,
    usecols=columnas_interaccion,
    nrows=100_000,
    low_memory=False
)

print(interactions.shape)

display(
    interactions.head()
)

(100000, 17)


,id,user_id,timestamp,url,mouse.position.x,mouse.position.y,mouse.clicks,mouse.clicks.left,scroll.absolute.x,scroll.absolute.y,scroll.relative.x,scroll.relative.y,keyboard,keyboard.alpha,keyboard.numeric,keyboard.function,keyboard.symbol
0,67892e1a85dffb571207298e,678927b385dffb57120720c3,1737043478562,https://www.repubblica.it/,0,0,False,False,0.0,400.0,100.0,400.0,False,False,False,False,False
1,67892e1a85dffb571207298f,678927b385dffb57120720c3,1737043478685,https://www.repubblica.it/,0,0,False,False,0.0,400.0,100.0,400.0,False,False,False,False,False
2,67892e1a85dffb5712072990,678927b385dffb57120720c3,1737043478810,https://www.repubblica.it/,0,0,False,False,0.0,400.0,100.0,400.0,False,False,False,False,False
3,67892e1a85dffb5712072991,678927b385dffb57120720c3,1737043478857,https://www.repubblica.it/,0,0,False,False,0.0,400.0,100.0,400.0,False,False,False,False,False
4,67892e1a85dffb5712072992,678927b385dffb57120720c3,1737043478966,https://www.repubblica.it/,0,0,False,False,0.0,400.0,100.0,400.0,False,False,False,False,False


In [5]:
interactions["datetime"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms",
    errors="coerce"
)

interactions[
    [
        "timestamp",
        "datetime"
    ]
].head()

,timestamp,datetime
0,1737043478562,2025-01-16 16:04:38.562
1,1737043478685,2025-01-16 16:04:38.685
2,1737043478810,2025-01-16 16:04:38.810
3,1737043478857,2025-01-16 16:04:38.857
4,1737043478966,2025-01-16 16:04:38.966


In [6]:
interactions = (
    interactions
    .sort_values(
        [
            "user_id",
            "timestamp"
        ]
    )
    .reset_index(drop=True)
)

In [7]:
print(
    "Usuarios:",
    interactions[
        "user_id"
    ].nunique()
)

print(
    "URLs:",
    interactions[
        "url"
    ].nunique()
)

Usuarios: 1
URLs: 39


In [8]:
interactions[
    "url"
].value_counts().head(20)

url
https://mail.google.com/                  20119
https://new.precisionconference.com/      14514
https://web.whatsapp.com/                 14197
https://www.overleaf.com/                  8038
http://giuseppe-desolda.ddns.net:1880/     6300
https://www.vodafone.it/                   5008
https://www.uniba.it/                      4940
https://www.youtube.com/                   4707
https://www.google.com/                    4500
https://cockpit-project.org/               2843
https://www.facebook.com/                  2512
https://www.repubblica.it/                 1505
https://scholar.google.it/                 1396
https://www.itzgeek.com/                   1360
https://www.ansa.it/                       1279
https://www.howtoforge.com/                 688
http://manpages.ubuntu.com/                 633
http://www.di.uniba.it/                     553
https://susy.mdpi.com/                      490
https://www.ubuntupit.com/                  480
Name: count, dtype: int64

In [9]:
interactions[
    "mouse.clicks"
].value_counts(
    dropna=False
)

mouse.clicks
False    85698
True     14302
Name: count, dtype: int64

In [10]:
interactions[
    "mouse.clicks.left"
].value_counts(
    dropna=False
)

mouse.clicks.left
False    85769
True     14231
Name: count, dtype: int64

In [11]:
click_count_sample = (
    interactions[
        "mouse.clicks"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)

click_count_sample

np.int64(14302)

In [12]:
interactions[
    "keyboard"
].value_counts(
    dropna=False
)

keyboard
False    89448
True     10552
Name: count, dtype: int64

In [13]:
keypress_count_sample = (
    interactions[
        "keyboard"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
)

keypress_count_sample

np.int64(10552)

In [14]:
keyboard_counts = {
    "alphabetic": (
        interactions["keyboard.alpha"]
        .fillna(False)
        .astype(bool)
        .sum()
    ),

    "numeric": (
        interactions["keyboard.numeric"]
        .fillna(False)
        .astype(bool)
        .sum()
    ),

    "function": (
        interactions["keyboard.function"]
        .fillna(False)
        .astype(bool)
        .sum()
    ),

    "symbol": (
        interactions["keyboard.symbol"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
}

keyboard_counts

{'alphabetic': np.int64(6444),
 'numeric': np.int64(53),
 'function': np.int64(4820),
 'symbol': np.int64(231)}

In [15]:
(interactions["scroll.absolute.y"] != 0).sum()

np.int64(26595)

In [16]:
interactions[
    "previous_scroll_y"
] = (
    interactions
    .groupby(
        "user_id"
    )[
        "scroll.absolute.y"
    ]
    .shift()
)

In [17]:
interactions[
    "scroll_changed"
] = (
    interactions[
        "scroll.absolute.y"
    ]
    !=
    interactions[
        "previous_scroll_y"
    ]
)

In [18]:
interactions.loc[
    interactions[
        "previous_scroll_y"
    ].isna(),
    "scroll_changed"
] = False

In [19]:
interactions[
    "scroll_changed"
].sum()

np.int64(1701)

In [20]:
interactions[
    "previous_timestamp"
] = (
    interactions
    .groupby(
        "user_id"
    )[
        "timestamp"
    ]
    .shift()
)

In [21]:
interactions[
    "inter_event_time_ms"
] = (
    interactions[
        "timestamp"
    ]
    -
    interactions[
        "previous_timestamp"
    ]
)

In [22]:
interactions[
    "inter_event_time_ms"
].describe(
    percentiles=[
        .50,
        .75,
        .90,
        .95,
        .99
    ]
)

count    9.999900e+04
mean     6.603987e+02
std      1.629747e+05
min      0.000000e+00
50%      6.800000e+01
75%      1.000000e+02
90%      1.010000e+02
95%      1.030000e+02
99%      1.320000e+02
max      5.144986e+07
Name: inter_event_time_ms, dtype: float64

In [23]:
click_events = interactions[
    interactions[
        "mouse.clicks"
    ]
    .fillna(False)
    .astype(bool)
].copy()

In [24]:
click_events[
    "inter_event_time_ms"
].describe()

count     14302.000000
mean        106.575933
std        1620.329411
min           0.000000
25%          57.000000
50%          99.000000
75%         101.000000
max      159236.000000
Name: inter_event_time_ms, dtype: float64

In [25]:
mean_pre_click_pause_s = (
    click_events[
        "inter_event_time_ms"
    ]
    .dropna()
    .mean()
    / 1000
)

mean_pre_click_pause_s

np.float64(0.10657593343588309)

In [26]:
clicks = interactions[
    interactions[
        "mouse.clicks"
    ]
    .fillna(False)
    .astype(bool)
].copy()

In [27]:
clicks = clicks.sort_values(
    [
        "user_id",
        "timestamp"
    ]
)

In [28]:
clicks["prev_x"] = (
    clicks
    .groupby("user_id")[
        "mouse.position.x"
    ]
    .shift()
)

clicks["prev_y"] = (
    clicks
    .groupby("user_id")[
        "mouse.position.y"
    ]
    .shift()
)

clicks["prev_ts"] = (
    clicks
    .groupby("user_id")[
        "timestamp"
    ]
    .shift()
)

In [29]:
clicks["delta_ms"] = (
    clicks["timestamp"]
    -
    clicks["prev_ts"]
)

In [30]:
clicks["distance_px"] = np.sqrt(
    (
        clicks[
            "mouse.position.x"
        ]
        -
        clicks["prev_x"]
    ) ** 2
    +
    (
        clicks[
            "mouse.position.y"
        ]
        -
        clicks["prev_y"]
    ) ** 2
)

In [31]:
MAX_TIME_MS = 1500
MAX_DISTANCE_PX = 20

In [32]:
clicks[
    "possible_repeated_click"
] = (
    (
        clicks["delta_ms"]
        <= MAX_TIME_MS
    )
    &
    (
        clicks["distance_px"]
        <= MAX_DISTANCE_PX
    )
)

In [33]:
clicks[
    "possible_repeated_click"
].value_counts()

possible_repeated_click
True     13445
False      857
Name: count, dtype: int64

In [34]:
interactions[
    "previous_url"
] = (
    interactions
    .groupby(
        "user_id"
    )[
        "url"
    ]
    .shift()
)

In [35]:
interactions[
    "url_changed"
] = (
    interactions["url"]
    !=
    interactions[
        "previous_url"
    ]
)

In [36]:
interactions.loc[
    interactions[
        "previous_url"
    ].isna(),
    "url_changed"
] = False

In [37]:
interactions[
    "url_changed"
].sum()

np.int64(222)

In [38]:
feature_validation = pd.DataFrame([
    {
        "variable_tesis":
            "click_count",

        "variable_origen":
            "mouse.clicks",

        "transformacion":
            "Suma de registros donde mouse.clicks=True",

        "estado":
            "VALIDADA"
    },

    {
        "variable_tesis":
            "keypress_count",

        "variable_origen":
            "keyboard",

        "transformacion":
            "Suma de registros donde keyboard=True",

        "estado":
            "VALIDADA"
    },

    {
        "variable_tesis":
            "scroll_count",

        "variable_origen":
            "scroll.absolute.y",

        "transformacion":
            "Cambios de posición vertical",

        "estado":
            "DERIVABLE"
    },

    {
        "variable_tesis":
            "mean_pre_action_pause_s",

        "variable_origen":
            "timestamp",

        "transformacion":
            "Promedio del intervalo inmediatamente anterior a clic",

        "estado":
            "DERIVABLE"
    },

    {
        "variable_tesis":
            "repeated_action_count",

        "variable_origen":
            "mouse.clicks + posición + timestamp",

        "transformacion":
            "Clics próximos temporal y espacialmente",

        "estado":
            "DERIVABLE_CON_REGLA"
    },

    {
        "variable_tesis":
            "navigation_steps",

        "variable_origen":
            "url",

        "transformacion":
            "Número de cambios consecutivos de URL",

        "estado":
            "PARCIAL"
    }
])

In [39]:
feature_validation

,variable_tesis,variable_origen,transformacion,estado
0,click_count,mouse.clicks,Suma de registros donde mouse.clicks=True,VALIDADA
1,keypress_count,keyboard,Suma de registros donde keyboard=True,VALIDADA
2,scroll_count,scroll.absolute.y,Cambios de posición vertical,DERIVABLE
3,mean_pre_action_pause_s,timestamp,Promedio del intervalo inmediatamente anterior...,DERIVABLE
4,repeated_action_count,mouse.clicks + posición + timestamp,Clics próximos temporal y espacialmente,DERIVABLE_CON_REGLA
5,navigation_steps,url,Número de cambios consecutivos de URL,PARCIAL


In [40]:
feature_validation.to_csv(
    INTERIM_DIR
    / "repo01_feature_validation.csv",
    index=False
)

In [41]:
aggregate_cols = [
    "middle.id",
    "middle.user_id",
    "middle.timestamp",
    "middle.url",

    "25.full.clicks.all.sum",
    "25.full.keys.all.sum",
    "25.full.scrolls.total",
    "25.full.event_times.avg",
    "25.full.idle.avg",
    "25.full.urls.changed"
]

In [42]:
aggregates_validation = pd.read_csv(
    AGGREGATES_PATH,
    usecols=aggregate_cols,
    nrows=100_000,
    low_memory=False
)

aggregates_validation.head()

,middle.id,middle.user_id,middle.timestamp,middle.url,25.full.clicks.all.sum,25.full.event_times.avg,25.full.idle.avg,25.full.keys.all.sum,25.full.scrolls.total,25.full.urls.changed
0,67892e3c85dffb5712072b91,678927b385dffb57120720c3,1737043507754,https://web.whatsapp.com/,0,0.0,0.0,0,0,0
1,67892e3c85dffb5712072b92,678927b385dffb57120720c3,1737043507857,https://web.whatsapp.com/,0,0.0,0.0,0,0,0
2,67892e3c85dffb5712072b98,678927b385dffb57120720c3,1737043508156,https://web.whatsapp.com/,0,0.0,0.0,1,0,0
3,67892e3c85dffb5712072b9b,678927b385dffb57120720c3,1737043508255,https://web.whatsapp.com/,0,0.0,0.0,2,1,0
4,678932e985dffb5712072ee8,678927b385dffb57120720c3,1737044703350,https://askubuntu.com/,0,0.0,0.0,0,0,0


In [43]:
aggregates_validation.describe().T

,count,mean,std,min,25%,50%,75%,max
middle.timestamp,100000.0,1.738087e+12,7.181602e+08,1.737044e+12,1.737878e+12,1.738063e+12,1.738145e+12,1.740997e+12
25.full.clicks.all.sum,100000.0,7.508000e-02,2.988376e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.000000e+00
25.full.event_times.avg,100000.0,2.720361e-01,1.403321e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.200000e+01
25.full.idle.avg,100000.0,3.807500e-03,1.242630e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.050000e+01
25.full.keys.all.sum,100000.0,1.850500e-01,4.923301e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.000000e+00
25.full.scrolls.total,100000.0,1.849400e-01,5.576381e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.600000e+01
25.full.urls.changed,100000.0,1.730000e-03,5.373115e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00


In [44]:
# Convertimos mouse.clicks en estado booleano
interactions["mouse_click_state"] = (
    interactions["mouse.clicks"]
    .fillna(False)
    .astype(bool)
)

# Estado anterior para cada usuario
interactions["previous_mouse_click_state"] = (
    interactions
    .groupby("user_id")["mouse_click_state"]
    .shift(fill_value=False)
)

# Consideramos un nuevo clic cuando cambia:
# False -> True
interactions["click_event"] = (
    interactions["mouse_click_state"]
    &
    ~interactions["previous_mouse_click_state"]
)

print("Registros mouse.clicks=True:")
print(
    interactions["mouse_click_state"].sum()
)

print("\nEventos de clic detectados:")
print(
    interactions["click_event"].sum()
)

Registros mouse.clicks=True:
14302

Eventos de clic detectados:
719


In [45]:
clicks_event = interactions[
    interactions["click_event"]
].copy()

clicks_event = clicks_event.sort_values(
    ["user_id", "timestamp"]
)

clicks_event["prev_x"] = (
    clicks_event
    .groupby("user_id")["mouse.position.x"]
    .shift()
)

clicks_event["prev_y"] = (
    clicks_event
    .groupby("user_id")["mouse.position.y"]
    .shift()
)

clicks_event["prev_timestamp"] = (
    clicks_event
    .groupby("user_id")["timestamp"]
    .shift()
)

clicks_event["delta_ms"] = (
    clicks_event["timestamp"]
    -
    clicks_event["prev_timestamp"]
)

clicks_event["distance_px"] = np.sqrt(
    (
        clicks_event["mouse.position.x"]
        -
        clicks_event["prev_x"]
    ) ** 2
    +
    (
        clicks_event["mouse.position.y"]
        -
        clicks_event["prev_y"]
    ) ** 2
)

MAX_TIME_MS = 1500
MAX_DISTANCE_PX = 20

clicks_event["possible_repeated_click"] = (
    (clicks_event["delta_ms"] <= MAX_TIME_MS)
    &
    (clicks_event["distance_px"] <= MAX_DISTANCE_PX)
)

clicks_event[
    "possible_repeated_click"
].value_counts()

possible_repeated_click
False    598
True     121
Name: count, dtype: int64

In [46]:
feature_validation.loc[
    feature_validation["variable_tesis"] == "click_count",
    "estado"
] = "DERIVABLE_VALIDADA_EN_MUESTRA"

feature_validation.loc[
    feature_validation["variable_tesis"] == "repeated_action_count",
    "estado"
] = "DERIVABLE_CON_REGLA_EXPERIMENTAL"

feature_validation.loc[
    feature_validation["variable_tesis"] == "mean_pre_action_pause_s",
    "estado"
] = "DERIVABLE_REQUIERE_VALIDACION"

In [47]:
feature_validation.to_csv(
    INTERIM_DIR
    / "repo01_feature_validation.csv",
    index=False
)